In [12]:
import cga_utils
import prompt_versions

import pandas as pd
import datetime
import energy

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:
from langchain.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

set_llm_cache(SQLiteCache(database_path=".langchain.ollama24_3_22311.db"))

In [32]:
from langchain_community.chat_models import ChatOllama
llm = ChatOllama(model="qwen3:4b", temperature = 0.0, top_p = 1, repeat_penalty=1, presence_penalty=0, frequency_penalty=0, num_predict=400)  # vagy bármely más elérhető Ollama modell

# V18 Base

In [ ]:
import pandas as pd
messages = prompt_versions.prompt_versions['V18']
messages = [(a, t.replace('/no_think','')) for (a,t)  in messages]
messages[-1] = ( messages[-1][0],  messages[-1][1] + "/no_think" ) 

start_global=datetime.datetime.now()
cga_utils.execute_dataset_predictions(llm, messages)
end_global=datetime.datetime.now()

resb = [{"prompt_version": "V18", "start_cluster": None, "end_cluster":None, "start_global":start_global, "end_global":end_global  }]
resb                                          

1774904358.4976587


0it [00:00, ?it/s]

b2786c1a-37de-4120-b03c-32bf5c81f157
What was the change in the amount for Appliances in 2019 from 2018? Success: -94.0 3.182s
<< 680-774 || [680.0, 774.0] || [680.0, 774.0] || True >>
fe11f001-3bfe-4089-8108-412676f0a780
What was the percentage change in the amount for Appliances in 2019 from 2018? failure: -12.14 good answer:  -12.14  1.661s
<< (680-774)/774 || [680.0, 774.0, -12.144702842377262] || [680.0, 774.0, 774.0] || True >>
5103aed0-b4e8-4fae-bf78-e2c9f4ba84cf
What is the difference between the domestic and international discount rates as at September 30, 2019? failure: 2.1 good answer:  2.1  1.587s
<< 4.00 - 1.90  || [4.0, 1.9] || [4.0, 1.9] || True >>
4dc8be43-d8d9-4b08-9ffd-9c19012361ce
What is the year on year percentage change in domestic discount rate between 2018 and 2019? failure: 6.67 good answer:  6.67  1.559s
<< (4.00 - 3.75)/3.75  || [3.75, 4.0, 6.666666666666667] || [4.0, 3.75, 3.75] || True >>
6c44a1a8-0785-43a0-90ab-7e21df2c57d9
What is the year on year percent

In [37]:
start = pd.Timestamp(resb[0]['start_global']).tz_localize("Europe/Budapest")
end = pd.Timestamp(resb[0]['end_global']).tz_localize("Europe/Budapest")
(wh_counter, wh_gauge_sum,wh_trap, delta ) = energy.fetch_gpu_wh(start, end,  "2s")
{"prompt_version": resb[0]["prompt_version"], "start": start, "end": end, "type": "global", "wh_counter": wh_counter, "wh_trap": wh_trap,  "delta": delta, "delta_us": delta, "delta_min": delta/(1000000*60), "avg_wh_counter": (wh_counter/delta)*1000000*3600 ,"avg_wh_trap": (wh_trap/delta)*1000000*3600 } 


{'prompt_version': 'V18',
 'start': Timestamp('2026-03-30 22:59:18.480723+0200', tz='Europe/Budapest'),
 'end': Timestamp('2026-03-30 23:11:07.128298+0200', tz='Europe/Budapest'),
 'type': 'global',
 'wh_counter': 82.0,
 'wh_trap': 81.87583333333333,
 'delta': 708647575.0,
 'delta_us': 708647575.0,
 'delta_min': 11.810792916666667,
 'avg_wh_counter': 416.56813684855973,
 'avg_wh_trap': 415.93735786085205}

In [36]:
resb


[{'prompt_version': 'V18',
  'start_cluster': None,
  'end_cluster': None,
  'start_global': datetime.datetime(2026, 3, 30, 22, 59, 18, 480723),
  'end_global': datetime.datetime(2026, 3, 30, 23, 11, 7, 128298)}]

# Iterations

In [39]:
runs=[('V23', 'res/e38_18.csv', 'res/errors_e38_18_cv1_4_2.csv', 11),
      ('V41', 'res/e38_23.csv', 'res/errors_e38_23_cv1_2_1.csv', 44),
      ('V43c', 'res/e38_41.csv', 'res/errors_e38_41_cv1_2_1.csv', 5 ),
      ('V29', 'res/e38_41.csv', 'res/errors_e38_41_cv1_2_1.csv', 22),
      ('V31', 'res/e38_29.csv', 'res/errors_e38_29_cv1_2_1.csv', 31),
      ('V44b', 'res/e38_29.csv', 'res/errors_e38_29_cv1_2_1.csv',15)      
     ]


In [10]:

for (prompt_version, results_file, errors_file, cluster_id) in runs:
    messages = prompt_versions.prompt_versions[prompt_version]
    messages = [(a, t.replace('/no_think','')) for (a,t)  in messages]
    messages[-1] = ( messages[-1][0],  messages[-1][1] + "/no_think" )
    print(prompt_version, "cluster Error:")
    start_cluster=datetime.datetime.now()
    cga_utils.calc_fix_mcnemar_performance(llm, messages, results_file, errors_file, cluster_id)
    end_cluster=datetime.datetime.now()
    print(prompt_version, "global Error:")
    start_global=datetime.datetime.now()
    cga_utils.execute_dataset_predictions(llm, messages)
    end_global=datetime.datetime.now()
    res.append({"prompt_version": prompt_version, "start_cluster":start_cluster, "end_cluster": end_cluster, "start_global":start_global, "end_global":end_global  })



V23 cluster Error:


0it [00:00, ?it/s]

cnt: 7
Base EM: 0.0
Base VM: 0.7142857142857143
Fix EM: 0.0
Fix VM: 0.7142857142857143
V23 global Error:
1774736768.39793


0it [00:00, ?it/s]

b2786c1a-37de-4120-b03c-32bf5c81f157
What was the change in the amount for Appliances in 2019 from 2018? Success: -94.0 1.729s
<< 680-774 || [680.0, 774.0, -94.0] || [680.0, 774.0] || True >>
fe11f001-3bfe-4089-8108-412676f0a780
What was the percentage change in the amount for Appliances in 2019 from 2018? Success: -12.14 1.704s
<< (680-774)/774 || [680.0, 774.0, -12.144702842377262] || [680.0, 774.0, 774.0] || True >>
5103aed0-b4e8-4fae-bf78-e2c9f4ba84cf
What is the difference between the domestic and international discount rates as at September 30, 2019? failure: 2.1 good answer:  2.1  1.416s
<< 4.00 - 1.90  || [4.0, 1.9, 2.1] || [4.0, 1.9] || True >>
4dc8be43-d8d9-4b08-9ffd-9c19012361ce
What is the year on year percentage change in domestic discount rate between 2018 and 2019? failure: 6.67 good answer:  6.67  1.837s
<< (4.00 - 3.75)/3.75  || [3.75, 4.0, 6.666666666666667] || [4.0, 3.75, 3.75] || True >>
6c44a1a8-0785-43a0-90ab-7e21df2c57d9
What is the year on year percentage change

...

0it [00:00, ?it/s]

cnt: 7
Base EM: 0.0
Base VM: 0.0
Fix EM: 0.5714285714285714
Fix VM: 0.7142857142857143
V31 global Error:
1774739697.4482994


0it [00:00, ?it/s]

b2786c1a-37de-4120-b03c-32bf5c81f157
What was the change in the amount for Appliances in 2019 from 2018? Success: -94.0 1.770s
<< 680-774 || [680.0, 774.0, -94.0] || [680.0, 774.0] || True >>
fe11f001-3bfe-4089-8108-412676f0a780
What was the percentage change in the amount for Appliances in 2019 from 2018? Success: -12.14 1.721s
<< (680-774)/774 || [680.0, 774.0, -12.144702842377262] || [680.0, 774.0, 774.0] || True >>
5103aed0-b4e8-4fae-bf78-e2c9f4ba84cf
What is the difference between the domestic and international discount rates as at September 30, 2019? failure: 2.1 good answer:  2.1  1.594s
<< 4.00 - 1.90  || [4.0, 1.9, 2.1] || [4.0, 1.9] || True >>
4dc8be43-d8d9-4b08-9ffd-9c19012361ce
What is the year on year percentage change in domestic discount rate between 2018 and 2019? failure: 6.67 good answer:  6.67  1.889s
<< (4.00 - 3.75)/3.75  || [3.75, 4.0, 6.666666666666667] || [4.0, 3.75, 3.75] || True >>
6c44a1a8-0785-43a0-90ab-7e21df2c57d9
What is the year on year percentage change

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [11]:
res 

[{'prompt_version': 'V18',
  'start_cluster': None,
  'end_cluster': None,
  'start_global': datetime.datetime(2026, 3, 28, 23, 25, 46, 941048),
  'end_global': datetime.datetime(2026, 3, 28, 23, 25, 50, 220728)},
 {'prompt_version': 'V23',
  'start_cluster': datetime.datetime(2026, 3, 28, 23, 26, 1, 937431),
  'end_cluster': datetime.datetime(2026, 3, 28, 23, 26, 8, 386517),
  'start_global': datetime.datetime(2026, 3, 28, 23, 26, 8, 386532),
  'end_global': datetime.datetime(2026, 3, 28, 23, 37, 45, 621257)},
 {'prompt_version': 'V41',
  'start_cluster': datetime.datetime(2026, 3, 28, 23, 37, 45, 621277),
  'end_cluster': datetime.datetime(2026, 3, 28, 23, 37, 58, 66638),
  'start_global': datetime.datetime(2026, 3, 28, 23, 37, 58, 66651),
  'end_global': datetime.datetime(2026, 3, 28, 23, 49, 39, 185486)},
 {'prompt_version': 'V43c',
  'start_cluster': datetime.datetime(2026, 3, 28, 23, 49, 39, 185504),
  'end_cluster': datetime.datetime(2026, 3, 28, 23, 49, 54, 747065),
  'start_gl

In [6]:
res = [{'prompt_version': 'V18',
  'start_cluster': None,
  'end_cluster': None,
  'start_global': datetime.datetime(2026, 3, 28, 23, 25, 46, 941048),
  'end_global': datetime.datetime(2026, 3, 28, 23, 25, 50, 220728)},
 {'prompt_version': 'V23',
  'start_cluster': datetime.datetime(2026, 3, 28, 23, 26, 1, 937431),
  'end_cluster': datetime.datetime(2026, 3, 28, 23, 26, 8, 386517),
  'start_global': datetime.datetime(2026, 3, 28, 23, 26, 8, 386532),
  'end_global': datetime.datetime(2026, 3, 28, 23, 37, 45, 621257)},
 {'prompt_version': 'V41',
  'start_cluster': datetime.datetime(2026, 3, 28, 23, 37, 45, 621277),
  'end_cluster': datetime.datetime(2026, 3, 28, 23, 37, 58, 66638),
  'start_global': datetime.datetime(2026, 3, 28, 23, 37, 58, 66651),
  'end_global': datetime.datetime(2026, 3, 28, 23, 49, 39, 185486)},
 {'prompt_version': 'V43c',
  'start_cluster': datetime.datetime(2026, 3, 28, 23, 49, 39, 185504),
  'end_cluster': datetime.datetime(2026, 3, 28, 23, 49, 54, 747065),
  'start_global': datetime.datetime(2026, 3, 28, 23, 49, 54, 747080),
  'end_global': datetime.datetime(2026, 3, 29, 0, 1, 54, 552729)},
 {'prompt_version': 'V29',
  'start_cluster': datetime.datetime(2026, 3, 29, 0, 1, 54, 552762),
  'end_cluster': datetime.datetime(2026, 3, 29, 0, 2, 4, 44867),
  'start_global': datetime.datetime(2026, 3, 29, 0, 2, 4, 44880),
  'end_global': datetime.datetime(2026, 3, 29, 0, 14, 45, 663111)},
 {'prompt_version': 'V31',
  'start_cluster': datetime.datetime(2026, 3, 29, 0, 14, 45, 663131),
  'end_cluster': datetime.datetime(2026, 3, 29, 0, 14, 57, 436949),
  'start_global': datetime.datetime(2026, 3, 29, 0, 14, 57, 436963),
  'end_global': datetime.datetime(2026, 3, 29, 0, 27, 25, 544397)},
 {'prompt_version': 'V44b',
  'start_cluster': datetime.datetime(2026, 3, 29, 0, 27, 25, 544417),
  'end_cluster': datetime.datetime(2026, 3, 29, 0, 27, 40, 159377),
  'start_global': datetime.datetime(2026, 3, 29, 0, 27, 40, 159391),
  'end_global': datetime.datetime(2026, 3, 29, 0, 40, 44, 164344)}]

In [7]:
pd.DataFrame(res)

,prompt_version,start_cluster,end_cluster,start_global,end_global
0,V18,NaT,NaT,2026-03-28 23:25:46.941048,2026-03-28 23:25:50.220728
1,V23,2026-03-28 23:26:01.937431,2026-03-28 23:26:08.386517,2026-03-28 23:26:08.386532,2026-03-28 23:37:45.621257
2,V41,2026-03-28 23:37:45.621277,2026-03-28 23:37:58.066638,2026-03-28 23:37:58.066651,2026-03-28 23:49:39.185486
3,V43c,2026-03-28 23:49:39.185504,2026-03-28 23:49:54.747065,2026-03-28 23:49:54.747080,2026-03-29 00:01:54.552729
4,V29,2026-03-29 00:01:54.552762,2026-03-29 00:02:04.044867,2026-03-29 00:02:04.044880,2026-03-29 00:14:45.663111
5,V31,2026-03-29 00:14:45.663131,2026-03-29 00:14:57.436949,2026-03-29 00:14:57.436963,2026-03-29 00:27:25.544397
6,V44b,2026-03-29 00:27:25.544417,2026-03-29 00:27:40.159377,2026-03-29 00:27:40.159391,2026-03-29 00:40:44.164344


In [30]:
res_e = []
for row in res:
    if row['start_cluster'] and row['end_cluster']:
        start = pd.Timestamp(row['start_cluster']).tz_localize("Europe/Budapest")
        end = pd.Timestamp(row['end_cluster']).tz_localize("Europe/Budapest")
        (wh_counter, wh_gauge_sum,wh_trap, delta ) = energy.fetch_gpu_wh(start, end,  "2s")
        res_e.append({"prompt_version": row["prompt_version"], "start": start, "end": end, "type": "cluster", "wh_counter": wh_counter, "wh_trap": wh_trap, "delta": delta,  "delta_us": delta, "delta_min": delta/(1000000*60), "avg_wh_counter": (wh_counter/delta)*1000000*3600,"avg_wh_trap": (wh_trap/delta)*1000000*3600 })
    if row['start_global'] and row['end_global']:
        start = pd.Timestamp(row['start_global']).tz_localize("Europe/Budapest")
        end = pd.Timestamp(row['end_global']).tz_localize("Europe/Budapest")
        (wh_counter, wh_gauge_sum,wh_trap, delta ) = energy.fetch_gpu_wh(start, end,  "2s")
        res_e.append({"prompt_version": row["prompt_version"], "start": start, "end": end, "type": "global", "wh_counter": wh_counter, "wh_trap": wh_trap,  "delta": delta, "delta_us": delta, "delta_min": delta/(1000000*60), "avg_wh_counter": (wh_counter/delta)*1000000*3600 ,"avg_wh_trap": (wh_trap/delta)*1000000*3600 }  )
pd.DataFrame(res_e)

,prompt_version,start,end,type,wh_counter,wh_trap,delta,delta_us,delta_min,avg_wh_counter,avg_wh_trap
0,V18,2026-03-28 23:25:46.941048+01:00,2026-03-28 23:25:50.220728+01:00,global,0.0,0.046111,3279680.0,3279680.0,0.054661,0.000000,50.614694
1,V23,2026-03-28 23:26:01.937431+01:00,2026-03-28 23:26:08.386517+01:00,cluster,1.0,0.710833,6449086.0,6449086.0,0.107485,558.218637,396.800415
2,V23,2026-03-28 23:26:08.386532+01:00,2026-03-28 23:37:45.621257+01:00,global,81.0,81.433333,697234725.0,697234725.0,11.620579,418.223576,420.460986
3,V41,2026-03-28 23:37:45.621277+01:00,2026-03-28 23:37:58.066638+01:00,cluster,1.0,1.410833,12445361.0,12445361.0,0.207423,289.264409,408.103871
4,V41,2026-03-28 23:37:58.066651+01:00,2026-03-28 23:49:39.185486+01:00,global,82.0,81.257222,701118835.0,701118835.0,11.685314,421.041320,417.227416
5,V43c,2026-03-28 23:49:39.185504+01:00,2026-03-28 23:49:54.747065+01:00,cluster,2.0,1.634167,15561561.0,15561561.0,0.259359,462.678519,378.046907
6,V43c,2026-03-28 23:49:54.747080+01:00,2026-03-29 00:01:54.552729+01:00,global,84.0,83.903611,719805649.0,719805649.0,11.996761,420.113402,419.631327
7,V29,2026-03-29 00:01:54.552762+01:00,2026-03-29 00:02:04.044867+01:00,cluster,1.0,0.946111,9492105.0,9492105.0,0.158202,379.262556,358.824518
8,V29,2026-03-29 00:02:04.044880+01:00,2026-03-29 00:14:45.663111+01:00,global,88.0,88.568611,761618231.0,761618231.0,12.693637,415.956429,418.644128
9,V31,2026-03-29 00:14:45.663131+01:00,2026-03-29 00:14:57.436949+01:00,cluster,1.0,1.225000,11773818.0,11773818.0,0.196230,305.763177,374.559892


In [45]:
for (prompt_version, results_file, errors_file, cluster_id) in runs:
    #print(prompt_version, errors_file, cluster_id)
    error_cluster_qids = list(pd.read_csv(errors_file).query(f'cluster == {cluster_id}')['qid'])
    print(prompt_version, len(error_cluster_qids))

V23 7
V41 8
V43c 10
V29 6
V31 7
V44b 7
